In [1]:
! pip install torch==2.9.0 transformers==4.57.3 matplotlib==3.10.0 pandas==2.2.2 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_ID = "Qwen/Qwen2.5-0.5B"
INST_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, mdl

base_tok, base_m = load(BASE_ID)
inst_tok, inst_m = load(INST_ID)

@torch.inference_mode()
def gen_base(prompt, max_new_tokens=80):
    x = base_tok(prompt, return_tensors="pt").to(base_m.device)
    y = base_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=base_tok.eos_token_id,
    )
    return base_tok.decode(y[0], skip_special_tokens=True)

@torch.inference_mode()
def gen_instruct(user_prompt, max_new_tokens=80):
    # Proper chat formatting for instruct model
    chat = [{"role": "user", "content": user_prompt}]
    prompt = inst_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = inst_tok(prompt, return_tensors="pt").to(inst_m.device)
    y = inst_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=inst_tok.eos_token_id,
    )
    out = inst_tok.decode(y[0], skip_special_tokens=True)
    # Trim the echoed prompt if present
    return out[len(inst_tok.decode(x["input_ids"][0], skip_special_tokens=True)) :].strip()

tests = [
    "How do I make a pizza?",
    "Write a haiku about summer.",
]

for p in tests:
    print("\n" + "="*90)
    print("PROMPT:", p)
    print("-"*90)
    print("BASE (pretrained):")
    print(gen_base(p))
    print("-"*90)
    print("INSTRUCT (fine-tuned):")
    print(gen_instruct(p))
